# 6.2 · Mini-batch K-Means / Mini-batch K-Means

> **课程定位 / Where this fits**
> 6.1 的 Lloyd 算法每轮都要扫**全部** n 个点——数据放不进内存、或流式持续到达时就行不通。Mini-batch K-Means 每轮只用一个**随机小批**增量更新质心: **恒定内存**、支持 `partial_fit` 在线/分块训练(接 5.15)。注意一个常见误解——在现代多核 sklearn 上, 全量 `KMeans` 已高度优化, **未必比 mini-batch 慢**; mini-batch 真正的价值是**内存与流式**, 而非单纯墙钟速度。
> Mini-batch K-Means uses random batches for constant memory and streaming (partial_fit). Note: on modern multicore sklearn, full KMeans is so optimised it's often as fast — the real win is memory/streaming, not wall-clock.

> 💡 **面试相关 / Interview-relevant**
> - "mini-batch K-Means 相比标准 K-Means 的取舍" ★★★★
> - "为什么用 mini-batch(内存/流式, 不一定是速度)" ★★★★
> - "质心更新的流式公式 / partial_fit" ★★★

---

## 学习目标 / Learning Objectives
1. mini-batch 更新规则(每个质心的流式均值)。
2. **真实卖点**: 恒定内存 + `partial_fit` 流式聚类。
3. 质量(inertia)代价有多小。
4. batch_size 的影响。

## 目录 / TOC
1. [mini-batch 更新规则 ⭐](#1)
2. [📦 数据: 大规模合成 blobs](#2)
3. [质量代价 + 流式 partial_fit ⭐](#3)
4. [batch_size 影响](#4)
5. [小结](#5)


<a id="1"></a>
## 1. mini-batch 更新规则 ⭐ / Update Rule

标准 K-Means 更新: 质心 = 全簇均值(扫全部点)。Mini-batch 每步只抽一个大小 $b$ 的随机批:
1. 把批内每点分配到最近质心。
2. 对每个被分到的点, **增量更新**其质心(带逐质心的学习率, 计数越多步长越小):
$$\boldsymbol\mu_k \leftarrow (1-\eta)\boldsymbol\mu_k + \eta\,\mathbf{x}, \qquad \eta = \frac{1}{\text{count}_k}$$

这正是**流式/在线均值**(每个质心维护已分配计数), 和 5.15 的增量学习同源。质心随批次抖动但快速收敛到接近全量解的位置。


<a id="2"></a>
## 2. 数据: 大规模合成 blobs / Large Synthetic Blobs

用 `make_blobs` 造一个**大规模**(20 万点)、明确分群的数据集, 好让速度差异显现。`make_blobs` 是聚类教学的标准合成器: 指定簇数、维度、离散度, 生成各向同性高斯团。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans, MiniBatchKMeans
sns.set_theme(style="whitegrid")

X, ytrue = make_blobs(n_samples=200_000, centers=8, n_features=10,
                      cluster_std=1.5, random_state=0)
print(f"大规模 blobs: {X.shape}, 8 个真实簇")
# 取前两维看一眼
fig, ax = plt.subplots(figsize=(6.5,5))
ax.scatter(X[:3000,0], X[:3000,1], c=ytrue[:3000], cmap="tab10", s=6, alpha=0.5)
ax.set_title("大规模 blobs(前 3000 点, 前 2 维): 8 个高斯团")
plt.tight_layout(); plt.show()


<a id="3"></a>
## 3. 质量代价 + 流式 partial_fit ⭐ / Quality Cost & Streaming

先比质量(inertia)和墙钟时间——会看到现代 sklearn 全量 KMeans 又快又准, mini-batch 的 inertia 仅略高。**真正的卖点在第二段**: `partial_fit` 让数据**一块块流入**, 内存恒定, 永不需一次装下全量。


In [ ]:
def timed_fit(model, name):
    t = time.perf_counter(); model.fit(X); dt = time.perf_counter()-t
    print(f"{name:<22} 训练 {dt:6.2f}s   inertia {model.inertia_:.3e}")
    return dt, model.inertia_

t1, i1 = timed_fit(KMeans(n_clusters=8, n_init=3, random_state=0), "标准 K-Means")
t2, i2 = timed_fit(MiniBatchKMeans(n_clusters=8, batch_size=2048, n_init=3, random_state=0), "Mini-batch K-Means")
print(f"\ninertia 仅高 {(i2-i1)/i1:.2%} (质量损失极小)")
print("⚠️ 注意墙钟: 现代 sklearn KMeans 用多核 Cython, 此处反而比 mini-batch 快——")
print("   '加速'是旧单线程时代的说法; 今天 mini-batch 的价值在内存与流式, 见下。")


In [ ]:
# 真实卖点: 流式 partial_fit, 数据分块到达, 内存恒定 / streaming partial_fit
mbk = MiniBatchKMeans(n_clusters=8, batch_size=2048, random_state=0)
n_chunks = 50
for chunk in np.array_split(X, n_chunks):    # 模拟 50 块依次流入(从不一次装全量)
    mbk.partial_fit(chunk)                    # 增量更新, 只持有当前块
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(ytrue, mbk.predict(X))
print(f"流式 partial_fit({n_chunks} 块) 后, 与真实簇的 ARI: {ari:.3f} (≈1 表示完整恢复了真实分群)")
print("(ARI=Adjusted Rand Index, 衡量聚类与真实标签的一致性, 0=随机 1=完美)")
print("→ 全程只持有一块数据, 内存恒定; 适合超大/无法载入内存/持续到达的数据")


<a id="4"></a>
## 4. batch_size 影响 / Effect of batch_size

`batch_size` 大 → 每步更准、更接近全量、但更慢; 小 → 快但质心抖动大、inertia 略高。常用几百到几千。


In [ ]:
sizes = [128, 512, 1024, 4096, 16384]
res = []
for b in sizes:
    t = time.perf_counter()
    m = MiniBatchKMeans(n_clusters=8, batch_size=b, n_init=3, random_state=0).fit(X)
    res.append((time.perf_counter()-t, m.inertia_))
times, inert = zip(*res)
fig, ax1 = plt.subplots(figsize=(7,4))
ax1.plot(sizes, times, "o-", color="tab:blue"); ax1.set_xlabel("batch_size"); ax1.set_xscale("log", base=2)
ax1.set_ylabel("训练时间 (s)", color="tab:blue")
ax2 = ax1.twinx(); ax2.plot(sizes, inert, "s--", color="tab:red"); ax2.set_ylabel("inertia", color="tab:red")
ax1.set_title("batch_size: 越大越准(inertia↓)但每步越慢"); ax1.legend(loc="upper left")
plt.tight_layout(); plt.show()
print("batch_size 在 每步速度↔质量(inertia) 间权衡; 几百~几千通常够好")


<a id="5"></a>
## 5. 小结 / Summary

```
Mini-batch K-Means: 每轮只用随机小批增量更新质心(流式均值, η=1/count_k)
真实卖点: 恒定内存 + partial_fit 流式/分块训练(数据放不进内存或持续到达)
inertia 仅略高(质量损失小); 墙钟未必更快——现代 sklearn KMeans 多核已极快
batch_size: 大→准但慢, 小→快但抖动; 几百~几千常用
其余(选K、缩放、球形假设)与标准 K-Means(6.1)相同
```

### 💡 面试速查
1. **mini-batch 用小批增量更新质心**; 卖点是**内存/流式**, 不一定是速度
2. **质心=流式均值**(每质心维护计数, 步长 1/count); 支持 **partial_fit**
3. **取舍**: inertia 略高几个百分点, 换恒定内存与在线能力
4. batch_size 控质量↔抖动; 其它假设同 K-Means

### 下一节
**6.3 层次聚类**——不需要预先指定 K, 自底向上合并(或自顶向下分裂), 输出一棵可切的树状图(dendrogram)。
